# Full-A/D causal sensitivity + robustness checks

Raises the causal endpoints from **n=30** (held-out A only) to **n=150**
(full A and D), and runs the D-sub-source robustness check.

Runtime → Run all. GPU needed for step 3 only; steps 4-6 are CPU.

**What this does NOT touch:** the frozen held-out-30 CF2 files. Everything
here writes to separate `*_fullAD.json` artifacts.


## 0. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Clone/pin, install, bind (HF cache stays OFF Drive)

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/urosavurdic/dpo-safety-representations.git'
REPO_DIR = '/content/dpo-safety-representations'
BRANCH = 'agent/c-quadrant-end-to-end-e0e2317a'
PINNED_COMMIT = 'ef8c86f193e769a8557293832cb2dc9c7c415586'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(['git', 'fetch', 'origin'], check=True)
subprocess.run(['git', 'checkout', PINNED_COMMIT], check=True)
print('checked out', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())


In [ ]:
!pip -q install -r requirements.txt
!pip -q install -U "bitsandbytes>=0.46.1"
!pip uninstall -y torchao || true
!nvidia-smi


In [ ]:
import os, glob

candidates = ['/content/drive/MyDrive/dpo_v2']
candidates += sorted(glob.glob('/content/drive/.shortcut-targets-by-id/*/dpo_v2'))
candidates += sorted(glob.glob('/content/drive/Shareddrives/*/dpo_v2'))
real_root = None
for c in candidates:
    if glob.glob(os.path.join(c, 'results', 'activations', '*_final.npy')):
        real_root = c; break
if real_root is None:
    raise SystemExit('No dpo_v2 folder with real activations found. Checked:\n  '
                     + '\n  '.join(candidates))
os.environ['DPO_DRIVE_ROOT'] = real_root
print('DPO_DRIVE_ROOT =', real_root)

from src.colab_persist import bind, status_line
info = bind(persist_hf_cache=False)   # judges are ~20 GB; keep them off Drive
print(status_line(info))
!python -m src.analysis.v2_pipeline status


## 2. HuggingFace auth (only needed if you run step 5, the re-judge)
Colab secret `HF_TOKEN`, notebook access ON. Token must belong to an account
that accepted the `google/gemma-2b` **and** `allenai/wildguard` licences.

In [ ]:
import os
try:
    from google.colab import userdata
    from huggingface_hub import login
    _tok = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = _tok
    login(token=_tok)
    print('HF login OK')
except Exception as e:
    print('HF NOT authenticated:', repr(e))
    print('Steps 3-4 and 6 still run. Only step 5 (re-judge) needs this.')


## 3. Full-A/D causal ablation — THE MAIN GPU STEP

Quadrants A and D in full (150 each, direction-estimation half included),
B/C skipped. Writes `causal_ablation_v2_{stage}_L24-28_fullAD.json`.

**M3 runs alone first as a smoke test** — this flag has never touched a real
model. Check the row count printed before letting the other three run.

In [ ]:
!python -m src.analysis.v2_pipeline causal --stage M3 --all-ad-sensitivity


In [ ]:
import json, os
p = 'results/raw/causal_ablation_v2_M3_L24-28_fullAD.json'
assert os.path.exists(p), 'smoke test FAILED - file not written'
rows = json.load(open(p))
from collections import Counter
print('rows:', len(rows), '(expect 900 = 300 x 3 conditions)')
print('by condition:', dict(Counter(r.get('stage') for r in rows)))
print('by quadrant :', dict(Counter(r.get('quadrant') for r in rows)))
assert len(rows) == 900, f'expected 900 rows, got {len(rows)} - STOP and report'
print('\nsmoke test PASSED - safe to run the other three branches')


In [ ]:
for st in ['M3_direct', 'M3_alt', 'M3_direct_alt']:
    get_ipython().system(f'python -m src.analysis.v2_pipeline causal --stage {st} --all-ad-sensitivity')


## 4. Direction-specificity at n=150 — CPU, no judge needed

The AD-vs-random McNemar on quadrant A (refusal) and quadrant D
(over-refusal side-effect). This is the power upgrade: n=150 instead of 30.

In [ ]:
for st in ['M3', 'M3_direct', 'M3_alt', 'M3_direct_alt']:
    f = f'results/raw/causal_ablation_v2_{st}_L24-28_fullAD.json'
    print('='*70); print(st); print('='*70)
    get_ipython().system(f'python -m src.analysis.summarize_causal_ablation --file {f}')
    for q, cat in [('A', 'refusal'), ('D', 'refusal')]:
        print(f'--- quadrant {q}, {cat}: ablated_AD vs ablated_random (direction-specificity) ---')
        get_ipython().system(
            f'python -m src.analysis.mcnemar_causal_ablation --file {f} '
            f'--conditions {st}_ablated_AD {st}_ablated_random --quadrant {q} --category {cat}')
    get_ipython().system(
        f'python -m src.analysis.bootstrap_causal_effect --file {f} --quadrant A --category refusal')


## 5. Re-judge (OPTIONAL, ~90-120 min) — continuous StrongREJECT at n=150

Only needed for the CF2 *continuous* sensitivity. Step 4 already gives you
the regex direction-specificity at n=150. Skip this if the session is tight.

In [ ]:
!python -m src.analysis.behavioral_judges \
  --response-manifest results/manifests/consolidated_judge.json \
  --from-results-dir results \
  --out-dir results/behavioral_judges_v2 \
  --run-live --scope confirmatory


In [ ]:
import glob
judged = sorted(glob.glob('results/behavioral_judges_v2/behavioral_judges_v2_*.json'))[-1]
print('Using:', judged)
!python -m src.analysis.confirmatory_behavioral_endpoints \
  --judged {judged} \
  --benchmark data/frozen_v2/benchmark_v2_20260826T212909Z.jsonl \
  --out results/summaries/confirmatory_endpoints.json


## 6. D sub-source robustness — CPU

Quadrant D = Alpaca + Dolly-15k + OASST1. Only OASST1 has no training
overlap. Re-estimates the direction on each sub-source and reports how far
it moves.

In [ ]:
!python -m src.analysis.direction_source_robustness


## Done

Send back:
1. Step 4's McNemar p-values (all four branches, quadrants A and D)
2. Step 6's `cos(d_full, d_OASST1)` table for all 9 stages
3. Step 5's confirmatory printout, if you ran it
